# 04 - Interior and Exterior Representations

A polyhedron can be described from the outside by linear inequalities or from the inside by points and directions that generate it. These are called its **H-representation** and **V-representation**. Moving between them is fundamental in polyhedral computation.

## Learning objectives

By the end of this notebook, you should be able to:

1. define halfspaces, supporting hyperplanes, vertices, recession directions, and rays;
2. read H- and V-representations and explain the geometry of each;
3. state the Minkowski–Weyl theorem and distinguish a polyhedron from a polytope;
4. convert a small bounded two-dimensional example in both directions;
5. explain why unbounded sets require directions as well as points; and
6. describe, at a conceptual level, how projection, homogenization, and polarity support general conversion algorithms.

## Prerequisites and terminology

We use polyhedra and convex combinations from Notebook 01, projection from Notebook 02, and conic combinations and polarity from Notebook 03. The word **interior representation** in the chapter title is a traditional synonym for a generator or V-representation; it does *not* refer to the topological interior of a set.


## Exterior description: the H-representation

An **H-representation** describes a polyhedron as the intersection of finitely many closed halfspaces:

$$
P=\{x\in\mathbb{R}^n\mid Ax\le b\}.
$$

Row $a_i^\top$ of $A$ gives the halfspace $a_i^\top x\le b_i$. Its boundary

$$
H_i=\{x\mid a_i^\top x=b_i\}
$$

is a hyperplane. If $P$ lies in the halfspace and touches $H_i$, then $H_i$ is a **supporting hyperplane** of $P$. At a point $x\in P$, inequality $i$ is **active** or **tight** if $a_i^\top x=b_i$; otherwise it is slack.

This view is called exterior because each inequality cuts away points from the ambient space. It is particularly natural for optimization models, whose business or physical rules are usually written as constraints.


## The geometric objects in a V-representation

A point $v\in P$ is a **vertex** or **extreme point** if it cannot be written as a nontrivial convex combination of two different points of $P$. Formally,

$$
v=\lambda x+(1-\lambda)y,
\quad x,y\in P,
\quad 0<\lambda<1
\quad\Longrightarrow\quad
x=y=v.
$$

A vector $r\ne0$ is a **recession direction** of $P$ if moving any nonnegative distance along it stays in $P$. A set $\{\mu r\mid\mu\ge0\}$ is a ray. For an H-represented nonempty polyhedron,

$$
\operatorname{rec}(P)
=\{r\mid x+\mu r\in P\ \forall x\in P,\ \mu\ge0\}
=\{r\mid Ar\le0\}.
$$

The right-hand side $b$ disappears because

$$
A(x+\mu r)=Ax+\mu Ar\le b
$$

whenever $Ax\le b$, $Ar\le0$, and $\mu\ge0$.

A cone is **pointed** if it contains no complete line through the origin. An **extreme ray** is a one-dimensional edge of a pointed cone—the directional analogue of a vertex. General generator lists may also contain redundant directions or both signs of a line direction.


## Interior description: the V-representation

A **V-representation** of a nonempty polyhedron uses finitely many points $v^1,\ldots,v^k$ and directions $r^1,\ldots,r^\ell$:

$$
P=\operatorname{conv}\{v^1,\ldots,v^k\}
 +\operatorname{cone}\{r^1,\ldots,r^\ell\}.
$$

The plus sign is a Minkowski sum. Equivalently, every $x\in P$ can be written as

$$
\begin{aligned}
x&=\sum_{i=1}^k\lambda_i v^i+\sum_{j=1}^\ell\mu_j r^j,\\
\sum_{i=1}^k\lambda_i&=1,\\
\lambda_i&\ge0,\qquad \mu_j\ge0.
\end{aligned}
$$

The first sum chooses a point in the convex hull of the listed points. The second adds any nonnegative amount of the listed unbounded directions.

Descriptions need not be minimal. An H-representation may contain redundant inequalities, and a V-representation may contain non-vertex points or redundant rays. Minimal descriptions are useful, but they are not required to define the same set.


## The Minkowski–Weyl theorem

The **Minkowski–Weyl theorem** states that, for a nonempty set $P\subseteq\mathbb{R}^n$, the following are equivalent:

1. $P$ is a polyhedron, meaning $P=\{x\mid Ax\le b\}$ for finite $A,b$;
2. $P$ is finitely generated, meaning
   $$
   P=\operatorname{conv}(V)+\operatorname{cone}(R)
   $$
   for finite point and direction sets $V,R$.

In short,

$$
\boxed{
\text{finite halfspaces}
\quad\Longleftrightarrow\quad
\text{finite points and directions}.
}
$$

A bounded polyhedron is a **polytope**. It has no nonzero recession direction, so

$$
P=\operatorname{conv}(V).
$$

Conversely, the convex hull of finitely many points is bounded. Thus, “polytope” can equivalently mean a bounded polyhedron or the convex hull of finitely many points.

The empty polyhedron has an H-representation but needs a separate convention on the V-side; we focus on nonempty polyhedra here.


## A bounded two-dimensional example

### From inequalities to vertices

Consider

$$
P=\left\{(x_1,x_2)\in\mathbb{R}^2\ \middle|\
\begin{aligned}
x_1&\ge0, & x_2&\ge0,\\
x_1&\le4, & x_2&\le3,\\
x_1+x_2&\le5
\end{aligned}
\right\}.
$$

The next cell stores these inequalities as $Ax\le b$ and displays the input data.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src import hrep_to_vertices_2d, vertices_to_hrep_2d, visualize_polyhedron

A = np.array(
    [
        [-1.0, 0.0],  # x1 >= 0
        [0.0, -1.0],  # x2 >= 0
        [1.0, 0.0],   # x1 <= 4
        [0.0, 1.0],   # x2 <= 3
        [1.0, 1.0],   # x1 + x2 <= 5
    ]
)
b = np.array([0.0, 0.0, 4.0, 3.0, 5.0]).reshape(-1, 1)

visualize_polyhedron(
    A,
    b,
    xlim=(-1, 5),
    ylim=(-1, 4),
    title="A polygon given by its H-representation",
)

In two dimensions, a vertex of a full-dimensional polygon occurs where at least two nonparallel supporting lines are active. This suggests a direct algorithm:

1. choose every pair of inequalities;
2. replace the chosen inequalities by equalities;
3. solve the resulting $2\times2$ linear system if the lines are not parallel;
4. keep the intersection only if it satisfies **all** inequalities;
5. remove duplicate points; and
6. order the remaining vertices around the polygon.

Checking all constraints is essential. For example, $x_1=4$ and $x_2=3$ intersect at $(4,3)$, but that point violates $x_1+x_2\le5$.

This pair-enumeration method is designed for transparent 2D examples. It does not handle arbitrary dimensions, and special care is needed for lower-dimensional sets where more than one equality may describe the affine hull.


#### H-to-V implementation for bounded 2D input

```{literalinclude} ../src/polyhedra/representations.py
:language: python
:linenos:
:start-after: "# --- REPRESENTATIONS:h_to_v_2d:start"
:end-before: "# --- REPRESENTATIONS:h_to_v_2d:end"
```


In [ ]:
vertices = hrep_to_vertices_2d(A, b)

print("Vertices in counterclockwise order:")
print(vertices)

The result is

$$
P=\operatorname{conv}\left\{
\begin{pmatrix}0\\0\end{pmatrix},
\begin{pmatrix}4\\0\end{pmatrix},
\begin{pmatrix}4\\1\end{pmatrix},
\begin{pmatrix}2\\3\end{pmatrix},
\begin{pmatrix}0\\3\end{pmatrix}
\right\}.
$$

Every point inside the polygon is a convex combination of these vertices. There are no ray generators because the polygon is bounded.


### From vertices to inequalities

Now suppose only a finite set of points is known. Their convex hull is the smallest convex set containing them. Each edge of a two-dimensional convex hull lies on a supporting line, and the entire polygon lies in one of the two associated halfspaces.

A convex-hull algorithm identifies the boundary vertices in cyclic order. For each directed edge, a perpendicular normal gives a candidate inequality; its sign is chosen so that every input point satisfies the same $\le$ inequality.

Duplicate points and points strictly inside the hull are harmless: they belong to the generated set but do not create boundary edges. Collinear input needs special handling because its hull is a line segment rather than a full-dimensional polygon.


#### V-to-H implementation for bounded 2D input

```{literalinclude} ../src/polyhedra/representations.py
:language: python
:linenos:
:start-after: "# --- REPRESENTATIONS:v_to_h_2d:start"
:end-before: "# --- REPRESENTATIONS:v_to_h_2d:end"
```


In [ ]:
A_from_vertices, b_from_vertices = vertices_to_hrep_2d(vertices)

print("Recovered inequalities [A | b]:")
print(np.hstack([A_from_vertices, b_from_vertices]))

max_violation = np.max(A_from_vertices @ vertices.T - b_from_vertices)
print(f"Largest inequality violation among the vertices: {max_violation:.2e}")

The recovered inequalities need not be textually identical to the original ones:

- rows may appear in a different order;
- multiplying an inequality by a positive scalar does not change its halfspace;
- a conversion algorithm may remove redundant input inequalities; and
- floating-point coefficients may differ by small numerical errors.

Representations should therefore be compared by the **sets they define**, not by direct array equality.


In [ ]:
visualize_polyhedron(
    A_from_vertices,
    b_from_vertices,
    xlim=(-1, 5),
    ylim=(-1, 4),
    title="The H-representation recovered from the vertices",
)

## Unbounded polyhedra

### Why vertices alone are not enough

Consider the nonnegative orthant

$$
P=\{x\in\mathbb{R}^2\mid x_1\ge0,\ x_2\ge0\}.
$$

Its only vertex is the origin, but the convex hull of the origin is just one point. The missing information is directional:

$$
r^1=\begin{pmatrix}1\\0\end{pmatrix},
\qquad
r^2=\begin{pmatrix}0\\1\end{pmatrix}.
$$

Hence

$$
P=
\operatorname{conv}\left\{\begin{pmatrix}0\\0\end{pmatrix}\right\}
+
\operatorname{cone}\left\{
\begin{pmatrix}1\\0\end{pmatrix},
\begin{pmatrix}0\\1\end{pmatrix}
\right\}.
$$


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
view_limit = 4.0
ax.fill_between([0, view_limit], 0, view_limit, alpha=0.15, label="P (clipped to the view)")
ax.scatter([0], [0], color="black", zorder=3, label="vertex")
ax.arrow(0, 0, 2.8, 0, width=0.035, length_includes_head=True, color="tab:red")
ax.arrow(0, 0, 0, 2.8, width=0.035, length_includes_head=True, color="tab:red", label="extreme rays")
ax.set_xlim(-0.5, view_limit)
ax.set_ylim(-0.5, view_limit)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("An unbounded V-representation")
ax.grid(True)
ax.legend()
plt.show()

### Lines and polyhedra without vertices

A recession cone can contain a **lineality space**

$$
\operatorname{lin}(P)=\operatorname{rec}(P)\cap(-\operatorname{rec}(P)),
$$

whose directions are feasible in both signs. For example, the strip

$$
S=\{(x_1,x_2)\mid0\le x_1\le1\}
$$

contains complete vertical lines and has no vertices. It can still be written as

$$
S=
\operatorname{conv}\left\{
\begin{pmatrix}0\\0\end{pmatrix},
\begin{pmatrix}1\\0\end{pmatrix}
\right\}
+
\operatorname{cone}\left\{
\begin{pmatrix}0\\1\end{pmatrix},
\begin{pmatrix}0\\-1\end{pmatrix}
\right\}.
$$

Including both $r$ and $-r$ generates unrestricted movement along their line. This is why the most general V-representation speaks of finite **generating directions**, not only extreme rays of a pointed cone.

The educational helper functions above intentionally restrict themselves to bounded 2D input. They are not general-purpose H/V conversion software.


## How general conversion works

### V-representation to H-representation: projection

Suppose points are columns of $V$ and directions are columns of $R$. Membership in the generated set means that there exist coefficients satisfying

$$
x=V\lambda+R\mu,
\qquad
\mathbf{1}^\top\lambda=1,
\qquad
\lambda,\mu\ge0.
$$

This is a polyhedron in the extended variables $(x,\lambda,\mu)$. Writing equalities as pairs of inequalities and projecting out $\lambda$ and $\mu$ leaves inequalities in $x$ alone. Fourier–Motzkin elimination proves existence of such an H-representation, although specialized methods are normally more efficient.

### H-representation to V-representation: homogenization

Starting from $P=\{x\mid Ax\le b\}$, construct the cone

$$
C=\left\{(x,t)\in\mathbb{R}^{n+1}\ \middle|\
Ax-bt\le0,\ t\ge0
\right\}.
$$

A cone generator with $t>0$ can be scaled to $t=1$ and then gives a point of $P$. A generator with $t=0$ satisfies $Ax\le0$ and gives a recession direction. This **homogenization** places points and directions into one conic object; polarity then exchanges its inequalities and generators.

Practical software uses algorithms such as the **double-description method**, beneath careful redundancy removal and exact or controlled arithmetic. Conversion may require exponentially many output inequalities or generators, so neither representation is uniformly compact.


## Faces, facets, and minimal descriptions

A valid inequality $a^\top x\le\beta$ defines the **face**

$$
F=\{x\in P\mid a^\top x=\beta\}
$$

when equality is attained. For a full-dimensional $n$-dimensional polyhedron, a face of dimension $n-1$ is a **facet**. In a polygon, facets are edges; in a three-dimensional polyhedron, facets are two-dimensional faces.

Vertices are zero-dimensional faces. Minimal H-representations of full-dimensional polyhedra typically keep one inequality per facet, while minimal V-representations of pointed polyhedra keep vertices and extreme rays. These correspondences explain why H/V conversion is also called facet/vertex enumeration.


## Check your understanding

1. What is the difference between a polyhedron and a polytope?
2. Why can $(4,3)$ fail to be a vertex of the example even though two boundary lines meet there?
3. What is $\operatorname{rec}(\{x\mid Ax\le b\})$?
4. Give a V-representation of the ray $[2,\infty)\subseteq\mathbb{R}$.
5. Why can equivalent H-representations contain different numbers of inequalities?

Brief answers: (1) a polytope is a bounded polyhedron; (2) it violates another inequality and is not feasible; (3) $\{r\mid Ar\le0\}$; (4) $\operatorname{conv}\{2\}+\operatorname{cone}\{1\}$; (5) descriptions can contain redundant or positively scaled inequalities.


## Takeaways and further reading

- H-representations intersect halfspaces; V-representations combine points and directions.
- The Minkowski–Weyl theorem says these describe the same class of nonempty polyhedra.
- Bounded polyhedra need only points; unbounded polyhedra need recession directions.
- Lines can be represented by including both signs of a direction.
- Small 2D conversions are geometrically simple, while general conversion can have exponential output size.

For proofs and polyhedral background, see [Komei Fukuda's Polyhedral Computation FAQ](https://people.inf.ethz.ch/fukudak/polyfaq/polyfaq.html) and the [Geometry of Optimization notes](https://www2.mathematik.tu-darmstadt.de/~paffenholz/data/preprints/geometry_of_optimization.pdf). The [cddlib documentation](https://github.com/cddlib/cddlib) describes software based on the double-description method.


## Next

A V-representation describes a polyhedron through generators. In integer optimization, we are especially interested in the convex hull generated by feasible lattice points. The next notebook introduces this **integer hull** and compares it with the original LP relaxation.

➡️ [05 - The Integer Hull](05-The-Integer-Hull.ipynb)
